# CIRRUS analysis: Steiner and ECCO on convective centres

Asked in the review of #65 (H1, #18), kept apart from the H1 scoring notebook. Is the January excess of
Steiner intensity centres (≥ 40 dBZ) a bright band? A second opinion comes from ECCO (Dixon and Romatschke
2022, `docs/sources/2022 - Dixon - …md`), which separates echo types by horizontal texture, not intensity,
with its OPERA 2 km parameters (`opera_ds.classify`, constants `ECCO_*`). ECCO has no winter validation and
its authors report bright band classified as convective (section 6): it is not a referee.

1. Share of Steiner intensity centres that ECCO calls stratiform (convectivity < 0.4), Belgium, January and
   September 2025, all CIRRUS frames, 2 km grid.
2. Maps of the January times with most of these disagreements: 2 km reflectivity with both
   classifications, and the frames 15 min before and after. Radar positions, from the single-radar
   volumes (`where` of `<country>/<node>/PVOL/…` in the ORD bucket), are drawn only if they have been
   downloaded (`DATA_DIR / "opera_pvol"`); they are there to see whether a ring is centred on a radar.
3. What is seen, described without a threshold: compact cells that move from frame to frame are
   convective; wide bands or rings that hardly move look like a bright band.

In [ ]:
%load_ext autoreload
%autoreload 2
from opera_ds.paths import DATA_DIR, RESULTS_DIR

import csv
import datetime as dt
import functools
import glob
from concurrent.futures import ProcessPoolExecutor

import h5py
import matplotlib.pyplot as plt
import numpy as np

from opera_ds.classify import ECCO_STRATIFORM_BELOW, intensity_centres_vs_ecco, steiner_ecco_fields
from opera_ds.odim_io import index_files, read_odim_composite

## Parameters

In [ ]:
MONTHS = {"2025-01": (dt.datetime(2025, 1, 1), dt.datetime(2025, 2, 1)),
          "2025-09": (dt.datetime(2025, 9, 1), dt.datetime(2025, 10, 1))}
N_TIMES = 15            # January times to map (10-20 asked in #65)
WORKERS = 8
OUT = RESULTS_DIR / "cirrus_analysis"
OUT.mkdir(parents=True, exist_ok=True)

## Files and Belgium mask on the 2 km grid

In [ ]:
files = index_files(str(DATA_DIR / "cirrus_bel" / "**" / "*@DBZH.h5"), r"@(\d{8}T\d{4})@", "%Y%m%dT%H%M")
ref = read_odim_composite(next(iter(files.values())))
r0, c0 = int(ref.where["crop_row0"]), int(ref.where["crop_col0"])
ny, nx = ref.values.shape
bel = np.load(DATA_DIR / "mask_BEL_1km.npy")[r0:r0 + ny, c0:c0 + nx]
bel2 = bel.reshape(ny // 2, 2, nx // 2, 2).all(axis=(1, 3))
np.save(OUT / "region_BEL_2km.npy", bel2)
print(f"crop {ny} x {nx} at 1 km; Belgium {bel2.sum()} px at 2 km")

## 1. Steiner intensity centres that ECCO calls stratiform

In [ ]:
def counts_for(month):
    a, b = MONTHS[month]
    times = [t for t in files if a <= t < b]
    work = functools.partial(intensity_centres_vs_ecco, region_coarse_path=str(OUT / "region_BEL_2km.npy"))
    with ProcessPoolExecutor(WORKERS) as ex:
        rows = list(ex.map(work, [files[t] for t in times], chunksize=16))
    for t, r in zip(times, rows):
        r["time"] = t.isoformat()
    with open(OUT / f"ecco_check_{month}.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["time"] + [k for k in rows[0] if k != "time"])
        w.writeheader(); w.writerows(rows)
    return rows

counts = {m: counts_for(m) for m in MONTHS}
for m, rows in counts.items():
    n = sum(r["intensity_centres"] for r in rows)
    s, x, i = (sum(r[k] for r in rows) for k in ("ecco_stratiform", "ecco_mixed", "ecco_inactive"))
    print(f"{m}: {len(rows)} frames, {n} intensity centres; ECCO stratiform {s / n:.1%}, mixed {x / n:.1%}, "
          f"not active {i / n:.1%}")

## 2. January times with most disagreements

The `N_TIMES` January times with the most Steiner intensity centres that ECCO calls stratiform.

In [ ]:
jan = sorted(counts["2025-01"], key=lambda r: -r["ecco_stratiform"])[:N_TIMES]
with open(OUT / "selected_january_times.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(jan[0]))
    w.writeheader(); w.writerows(jan)
for r in jan:
    print(r["time"], r["ecco_stratiform"], "of", r["intensity_centres"], "intensity centres")
print(f"{len({r['time'][:10] for r in jan})} distinct days")

Radar positions: download one single-radar file per radar and per day of the selected times (a separate
step, the notebook does not access the network), then rerun the cells below:

```
uv run python scripts/fetch_pvol_positions.py --times-csv results/cirrus_analysis/selected_january_times.csv \
    --countries BE NL LU DE FR --out data/opera_pvol
```

In [ ]:
def radar_positions():
    """{(day, node): (lon, lat)} from the 'where' group of the downloaded single-radar files."""
    out = {}
    for p in glob.glob(str(DATA_DIR / "opera_pvol" / "*" / "*.h5")):
        parts = p.replace("\\", "/").split("/")
        day, node = parts[-2], parts[-1].split("@")[0]
        with h5py.File(p, "r") as f:
            w = f["where"].attrs
            out[(day, node)] = (float(w["lon"]), float(w["lat"]))
    return out

radars = radar_positions()
print(f"radar positions available: {len(radars)} (day, radar) pairs" if radars else "none (files not downloaded)")

In [ ]:
from pyproj import Proj

proj = Proj(ref.where["projdef"])
x_ul, y_ul = proj(float(ref.where["UL_lon"]), float(ref.where["UL_lat"]))

def to_px(lon, lat, scale_m=2000.0):
    x, y = proj(lon, lat)
    return (x - x_ul) / scale_m, (y_ul - y) / scale_m

def frame(t):
    return steiner_ecco_fields(read_odim_composite(files[t]).to_field(0.0)) if t in files else None

for r in jan:
    t = dt.datetime.fromisoformat(r["time"])
    panels = [(t - dt.timedelta(minutes=15), "−15 min"), (t, "time t"), (t + dt.timedelta(minutes=15), "+15 min")]
    fig, axes = plt.subplots(2, 3, figsize=(15, 9.5))
    for j, (tt, label) in enumerate(panels):
        f = frame(tt)
        for i in range(2):
            ax = axes[i, j]
            ax.set_xticks([]); ax.set_yticks([])
            if f is None:
                ax.set_title(f"{tt:%d/%m %H:%M} missing"); continue
            ax.imshow(np.where(f["echo"], f["dbz"], np.nan), cmap="viridis", vmin=0, vmax=55)
            ax.contour(bel2, levels=[0.5], colors="white", linewidths=0.8)
            if i == 0:
                ax.contour(f["convective"], levels=[0.5], colors="red", linewidths=0.8)
                yy, xx = np.nonzero(f["by_intensity"])
                ax.scatter(xx, yy, s=1, c="red")
                ax.set_title(f"{tt:%d/%m %H:%M} ({label}) Steiner: centres ≥ 40 dBZ, convective area")
            else:
                c = f["convectivity"]
                ax.contour(np.nan_to_num(c, nan=0) >= 0.5, levels=[0.5], colors="orange", linewidths=0.8)
                ax.contour(np.nan_to_num(c, nan=1) < ECCO_STRATIFORM_BELOW, levels=[0.5], colors="cyan", linewidths=0.6)
                ax.set_title(f"ECCO: convective (orange), stratiform (cyan)")
            for (day, node), (lon, lat) in radars.items():
                if day != f"{t:%Y%m%d}":
                    continue
                px, py = to_px(lon, lat)
                if 0 <= px < f["dbz"].shape[1] and 0 <= py < f["dbz"].shape[0]:
                    ax.plot(px, py, "k^", ms=6)
                    ax.annotate(node, (px, py), color="black", fontsize=7)
    fig.tight_layout()
    fig.savefig(OUT / f"disagreement_{t:%Y%m%dT%H%M}.png", dpi=90)
    plt.close(fig)
print(f"maps saved in {OUT}")

## 3. What is seen

To fill after looking at the maps in `results/cirrus_analysis/`: compact cells that move between −15 min,
t and +15 min (convective), or wide bands and rings that hardly move (bright band), and whether rings are
centred on a radar. No threshold.